# Assignment 6: Attention (please!)

---

## Task 1) Thesis Title Classification

In this assignment, we'll again rely on the theses dataset and want to classify whether a thesis is bachelor or master.
Update your B.Sc. / M.Sc. thesis title classification model from the previous assignment and integrate the attention mechanism.
Therefore, implement the `dot product attention` and check how it affects the training and performance for this task.
In case you want to start fresh, we provide some boiler plate code of a base RNN classification model as well as ready-to-go data loading.
The basic setup as well as some code and steps can be reused from your solution for the RNN tasks.

### Data

Download the `theses.csv` data set from the `Supplemental Materials` in the `Files` section of our Microsoft Teams group.
This dataset consists of approx. 3,000 theses topics chosen by students in the past.
Here are some examples of the file content:

```
27.10.94;14.07.95;1995;intern;Diplom;DE;Monte Carlo-Simulation für ein gekoppeltes Round-Robin-System;
04.11.94;14.03.95;1995;intern;Diplom;DE;Implementierung eines Testüberdeckungsgrad-Analysators für RAS;
01.11.20;01.04.21;2021;intern;Bachelor;DE;Landessprachenerkennung mittels X-Vektoren und Meta-Klassifikation;
```

*In this Jupyter Notebook, we will provide the steps to solve this task and give hints via functions & comments. However, code modifications (e.g., function naming, arguments) and implementation of additional helper functions & classes are allowed. The code aims to help you get started.*

---

In [1]:
# Dependencies
import os
import re
import tqdm
import string
import numpy as np
import pandas as pd
import sklearn.metrics as sklearn_metrics
from sklearn.model_selection import StratifiedKFold

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

### Prepare the Data

1.1 Spend some time on preparing the dataset. It may be helpful to lower-case the data and to filter for German titles. The format of the CSV-file should be:

```
Anmeldedatum;Abgabedatum;JahrAkademisch;Art;Grad;Sprache;Titel;Abstract
```

1.2 Create the vocabulary from the prepared dataset. You'll need it for the modeling part such as nn.Embedding.

1.3 Filter out all diploma theses; they might be too easy to spot because they only cover "old" topics.

1.4 Create a PyTorch Dataset class which handles your tokenized data with respect to input and (class) labels.

In [2]:
def load_theses_dataset(filepath):
    """Loads all theses instances and returns them as a dataframe."""
    ### YOUR CODE HERE
    
    return pd.read_csv(filepath, header=0, sep=";")
    
    ### END YOUR CODE

In [3]:
def preprocess(dataframe):
    """Preprocesses and tokenizes the given theses titles for further use."""
    ### YOUR CODE HERE
    
    def _preprocss_fn(text):
        remove_digits = str.maketrans(string.digits, ' '*len(string.digits))
        remove_pun = str.maketrans(string.punctuation, ' '*len(string.punctuation))
        text = text.translate(remove_digits)
        text = text.translate(remove_pun)
        text = re.sub(' {2,}', ' ', text)
        return text.lower()
    
    dataframe = dataframe.copy()
    
    # Remove punctuation, digits and lowercase titles
    dataframe["Titel"] = dataframe["Titel"].apply(lambda s: _preprocss_fn(s))

    # Filter out empty and short titles
    dataframe = dataframe[dataframe["Titel"].str.len() > 4]

    # Reset index of dataframe
    dataframe = dataframe.reset_index(drop=True)

    # Simple tokenization of titles
    dataframe["tokenized"] = [title.split() for title in dataframe["Titel"].values]

    return dataframe

    ### END YOUR CODE

In [4]:
# Load and preprocess dataset
dataframe_all = load_theses_dataset("data/theses2022.csv")
dataframe_all = dataframe_all[dataframe_all["Sprache"] == "DE"]
dataframe_all = preprocess(dataframe_all)

# Convert labels to integer
LABEL2IDX = {"Bachelor": 0, "Master": 1, "Diplom": 2}
dataframe_all["label"] = dataframe_all["Grad"].apply(lambda l: LABEL2IDX[l])

# Filter out `Diplom`
dataframe_diplom = dataframe_all[dataframe_all["Grad"] == "Diplom"]
dataframe = dataframe_all[dataframe_all["Grad"] != "Diplom"]

# Check number of samples and label distribution
print(f"Num theses (overall): {len(dataframe_all)}")
print(f"Num theses (w/o diplom): {len(dataframe)}")
print(f"Num theses (diplom): {len(dataframe_diplom)}")
print()
print(dataframe_all["Grad"].value_counts())

Num theses (overall): 2982
Num theses (w/o diplom): 2126
Num theses (diplom): 856

Grad
Bachelor    1667
Diplom       856
Master       459
Name: count, dtype: int64


In [ ]:
### Notice: Think about padding tokens for batch sizes > 1
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("dbmdz/bert-base-german-cased")

# vocab = set()
# vocab.add("<pad>")

# # For a more realistic application, we have to deal with unknown tokens
# # that were not present in the training corpus. However, for the sake
# # of clarity, we add all possible tokens from our dataset.
# # vocab.add("<unk>")

# # Prepare vocabulary
# for s in dataframe_all.tokenized:
#     vocab.update(s)

# vocab_size = len(vocab)

# word2idx = {w: idx for (idx, w) in enumerate(sorted(vocab))}
# idx2word = {idx: w for (idx, w) in enumerate(sorted(vocab))}

# print(f"Vocabulary size: {vocab_size}")

ModuleNotFoundError: No module named 'transformers'

In [6]:
### PyTorch dataset for our thesis classification task

class ThesisClassificationDataset(Dataset):
    def __init__(self, dataset, labels, word2idx):
        self.data, self.labels = [], []
        for tokens, label in zip(dataset, labels):
            # Create inputs; map tokens to ids
            self.data.append(torch.stack([
                torch.tensor(word2idx[w], dtype=torch.long) for w in tokens
            ]))

            # Create labels; already an integer
            self.labels.append(label)


    def __len__(self):
        return len(self.data)


    def __getitem__(self, idx):
        # Returns one input and label sample
        return self.data[idx], self.labels[idx]

### Train and Evaluate

2.1 Implement the dot product attention mechanism and integrate it into your RNN classification model.

2.2 Train and evaluate your models with a train-test-split (or optional 5-fold cross-validation).

2.3 Assemble a table: Recall/Precision/F1 measure for RNN classification with and without attention. Do your results improve w.r.t. your old model?

2.4 Can you find certain words that receive high attention weights regarding the decision?

In [ ]:
### TODO: 2.1 Implement RNN classifier (nn.Module)
### Notice: Think about padding for batch sizes > 1
### Notice: 'torch.nn.utils.rnn' provides functionality
### Notice: Here you can integrate the attention mechanism

### YOUR CODE HERE

from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence, pad_sequence

class GRU_Classifier(nn.Module):
    def __init__(self, num_embeddings, embedding_dim, hidden_dim, num_classes,
                 with_attention=False):
        super(GRU_Classifier, self).__init__()
        self.with_attention = with_attention

        self.embedding = nn.Embedding(
            num_embeddings=num_embeddings, 
            embedding_dim=embedding_dim
        )

        self.rnn = nn.GRU(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            bidirectional=False,
            num_layers=1
        )

        if self.with_attention:
            self.attention = DotProductAttention(
                q_dim=hidden_dim,
                k_dim=hidden_dim,
                v_dim=hidden_dim,
                batch_first=False
            )

        self.fc = nn.Linear(hidden_dim, num_classes)

    
    def forward(self, X, lengths, hidden=None):
        embeddings = self.embedding(X)

        # Packed squence helps avoid unneccsary computation
        packed_seq = pack_padded_sequence(embeddings, lengths)

        outputs, hidden_states = self.rnn(packed_seq , hidden)

        # If tuple (h_n, c_n) containts cell state c_n then select h_n
        if isinstance(hidden_states, tuple):
            hidden = hidden_states[0]
        else:
            hidden = hidden_states

        if self.with_attention:
            # Padding for packed sequences
            padded_seq , lens = pad_packed_sequence(outputs)

            # Get top hidden states and add seq_len 1 for attention
            hidden = hidden[-1].unsqueeze(0)

            # Apply dot product attention
            clf_input, weights = self.attention(hidden, padded_seq, padded_seq)
        else:
            clf_input, weights = hidden, None

        # Apply classifier with hidden states
        logits = self.fc(clf_input.squeeze(0))

        return logits, hidden_states, weights
    

class DotProductAttention(nn.Module):
    def __init__(self, q_dim, k_dim, v_dim, batch_first=False):
        super().__init__()
        self.batch_first = batch_first
        self.scale = np.sqrt(k_dim)
        self.softm = nn.Softmax(dim=2)

        # Linear projection to same dimension
        self.W_q = nn.Linear(q_dim, k_dim)
        self.W_k = nn.Linear(q_dim, k_dim)
        self.W_v = nn.Linear(k_dim, v_dim)


    def forward(self, x_q, x_k, x_v):
        Q = self.W_q(x_q) # Q = x_q
        K = self.W_k(x_k) # K = x_k
        V = self.W_v(x_v) # V = x_v
        A, attn_weights = self.scaled_dp_attention(Q, K, V)
        return A, attn_weights
    

    def scaled_dp_attention(self, q, k, v):
        # Handling shape format
        if not self.batch_first:
            k = k.permute(1, 0, 2)
            q = q.permute(1, 0, 2)
            v = v.permute(1, 0, 2)

        # Apply attention with queries, keys, and values
        attn_weights = torch.matmul(q, k.transpose(1, 2))
        attn_weights = attn_weights / self.scale
        attn_weights = self.softm(attn_weights)
        A = torch.matmul(attn_weights, v)

        if not self.batch_first:
            A = A.permute(1, 0, 2)
            attn_weights = attn_weights.permute(1, 0, 2)
        return A, attn_weights


class SequencePadder():
    def __init__(self, symbol) -> None:
        self.symbol = symbol

    def __call__(self, batch):
        sorted_batch = sorted(batch, key=lambda x: x[0].size(0), reverse=True)
        sequences = [x[0] for x in sorted_batch]
        labels = [x[1] for x in sorted_batch]
        padded = pad_sequence(sequences, padding_value=self.symbol)
        lengths = torch.LongTensor([len(x) for x in sequences])
        return padded, torch.LongTensor(labels), lengths


### END YOUR CODE

## Erklärung des Attention-Mechanismus

Die folgende Implementierung zeigt einen **Dot Product Attention-Mechanismus** für RNN-basierte Textklassifikation. Der Attention-Mechanismus ermöglicht es dem Modell, sich auf relevante Teile der Eingabesequenz zu "fokussieren".

### Hauptkomponenten:

1. **GRU_Classifier**: Erweiterte RNN-Klassifikator mit optionaler Attention
2. **DotProductAttention**: Implementierung des Scaled Dot-Product Attention
3. **SequencePadder**: Hilfsfunktion für variable Sequenzlängen

### Attention-Grundidee:
Anstatt nur den finalen Hidden State zu verwenden, betrachtet das Modell **alle** Hidden States der Sequenz und lernt automatisch, welche am wichtigsten für die Klassifikation sind.

### 1. GRU_Classifier mit Attention

```python
class GRU_Classifier(nn.Module):
    def __init__(self, ..., with_attention=False):
        ...
        if self.with_attention:
            self.attention = DotProductAttention(...)
```

**Architektur-Erweiterung:**
- **Basis-RNN**: Verwendet GRU für Sequenzverarbeitung
- **Optionale Attention**: Flag `with_attention` aktiviert Attention-Mechanismus
- **Flexible Klassifikation**: Kann mit oder ohne Attention trainiert werden

**Forward-Pass mit Attention:**
```python
if self.with_attention:
    padded_seq, lens = pad_packed_sequence(outputs)
    hidden = hidden[-1].unsqueeze(0)  # Query: finale Hidden State
    clf_input, weights = self.attention(hidden, padded_seq, padded_seq)
else:
    clf_input, weights = hidden, None
```

**Attention-Workflow:**
1. **Unpack Sequences**: Konvertiert packed sequences zurück für Attention
2. **Query Definition**: Finaler Hidden State wird zur Query
3. **Attention Computation**: Berechnet Attention-Weights und gewichtete Summe
4. **Classification**: Verwendet Attention-Output für finale Klassifikation

### 2. DotProductAttention-Mechanismus

```python
class DotProductAttention(nn.Module):
    def __init__(self, q_dim, k_dim, v_dim, batch_first=False):
        self.scale = np.sqrt(k_dim)  # Skalierungsfaktor
        self.W_q = nn.Linear(q_dim, k_dim)  # Query-Projektion
        self.W_k = nn.Linear(q_dim, k_dim)  # Key-Projektion  
        self.W_v = nn.Linear(k_dim, v_dim)  # Value-Projektion
```

**Komponenten-Erklärung:**
- **Query (Q)**: "Was suche ich?" - Finaler Hidden State
- **Keys (K)**: "Womit vergleiche ich?" - Alle Hidden States der Sequenz
- **Values (V)**: "Was extrahiere ich?" - Alle Hidden States der Sequenz
- **Scaling**: Verhindert sehr große Dot-Products (√d_k normalisiert)

**Mathematische Formeln:**
```
Q = W_q × x_q    (Query-Transformation)
K = W_k × x_k    (Key-Transformation)
V = W_v × x_v    (Value-Transformation)

Attention(Q,K,V) = softmax(QK^T/√d_k)V
```

**Intuitive Erklärung:**
1. **Ähnlichkeit**: Q·K^T berechnet Ähnlichkeit zwischen Query und allen Keys
2. **Normalisierung**: Softmax konvertiert zu Wahrscheinlichkeitsverteilung
3. **Gewichtung**: Summiert Values basierend auf Attention-Weights

### 3. Scaled Dot-Product Attention Algorithmus

```python
def scaled_dp_attention(self, q, k, v):
    # Schritt 1: Tensor-Umformung für batch_first Format
    if not self.batch_first:
        k = k.permute(1, 0, 2)  # (seq_len, batch, dim) → (batch, seq_len, dim)
        q = q.permute(1, 0, 2)
        v = v.permute(1, 0, 2)
    
    # Schritt 2: Attention-Scores berechnen
    attn_weights = torch.matmul(q, k.transpose(1, 2))  # Q × K^T
    
    # Schritt 3: Skalierung
    attn_weights = attn_weights / self.scale  # ÷ √d_k
    
    # Schritt 4: Softmax-Normalisierung
    attn_weights = self.softmax(attn_weights)
    
    # Schritt 5: Gewichtete Summe
    A = torch.matmul(attn_weights, v)  # Attention_weights × V
    
    return A, attn_weights
```

**Schritt-für-Schritt-Erklärung:**

**Schritt 1**: **Tensor-Umformung**
- PyTorch RNNs verwenden `(seq_len, batch, dim)`-Format
- Attention benötigt `(batch, seq_len, dim)`-Format
- `permute()` reorganisiert Dimensionen

**Schritt 2**: **Attention-Scores**
- `torch.matmul(q, k.transpose(1, 2))` berechnet alle Ähnlichkeiten
- Query-Dimension: `(batch, 1, dim)`
- Key-Dimension: `(batch, seq_len, dim)`
- Ergebnis: `(batch, 1, seq_len)` - Scores für jede Position

### 4. Praktisches Beispiel: Attention bei Thesis-Klassifikation

**Beispiel-Titel**: "Implementierung eines maschinellen Lernalgorithmus"

**Ohne Attention:**
- Nur finaler Hidden State wird verwendet
- Verliert möglicherweise wichtige Informationen aus frühen Wörtern
- Klassifikation basiert auf "Lernalgorithmus" (letztes relevantes Wort)

**Mit Attention:**
- Alle Wörter werden gewichtet betrachtet
- Mögliche Attention-Weights:
  ```
  "Implementierung": 0.1    (weniger wichtig)
  "eines":          0.05   (weniger wichtig)  
  "maschinellen":   0.4    (sehr wichtig für Master-Thesis)
  "Lernalgorithmus": 0.45  (sehr wichtig für Master-Thesis)
  ```

**Attention-Berechnung:**
```python
# Query: Finaler Hidden State (repräsentiert gesamten Kontext)
# Keys: Alle Hidden States (jedes Wort)
# Values: Alle Hidden States (zu gewichtende Informationen)

# Attention-Weights zeigen Wichtigkeit jedes Wortes
# Gewichtete Summe kombiniert alle relevanten Informationen
```

**Vorteile:**
- **Interpretierbarkeit**: Visualisierung welche Wörter wichtig sind
- **Bessere Performance**: Nutzt alle verfügbaren Informationen
- **Robustheit**: Weniger anfällig für Informationsverlust

### 5. Technische Implementierungsdetails

#### SequencePadder Funktionalität:
```python
def __call__(self, batch):
    sorted_batch = sorted(batch, key=lambda x: x[0].size(0), reverse=True)
    padded = pad_sequence(sequences, padding_value=self.symbol)
    lengths = torch.LongTensor([len(x) for x in sequences])
```

**Zweck**: Verarbeitet Batches mit unterschiedlichen Sequenzlängen
- **Sortierung**: Längste Sequenzen zuerst (für packed sequences)
- **Padding**: Auffüllen kürzerer Sequenzen mit `<pad>`-Tokens
- **Längen-Tracking**: Originale Längen für korrekte Attention-Berechnung

#### Attention-Integration im Forward-Pass:
```python
# 1. Standard RNN-Verarbeitung
packed_seq = pack_padded_sequence(embeddings, lengths)
outputs, hidden_states = self.rnn(packed_seq, hidden)

# 2. Attention-spezifische Verarbeitung
if self.with_attention:
    padded_seq, lens = pad_packed_sequence(outputs)  # Unpack für Attention
    hidden = hidden[-1].unsqueeze(0)                # Query vorbereiten
    clf_input, weights = self.attention(hidden, padded_seq, padded_seq)
```

#### Dimensionen-Tracking:
- **Embeddings**: `(seq_len, batch, embedding_dim)`
- **RNN Output**: `(seq_len, batch, hidden_dim)`
- **Attention Query**: `(1, batch, hidden_dim)`
- **Attention Output**: `(1, batch, hidden_dim)`
- **Final Classification**: `(batch, num_classes)`

### 6. Vergleich: Mit vs. Ohne Attention

| Aspekt | Ohne Attention | Mit Attention |
|--------|---------------|---------------|
| **Information** | Nur finaler Hidden State | Alle Hidden States gewichtet |
| **Interpretierbarkeit** | Schwarz-Box | Attention-Weights visualisierbar |
| **Komplexität** | Einfach | Höher (zusätzliche Parameter) |
| **Performance** | Gut für kurze Sequenzen | Besser für längere Sequenzen |
| **Training** | Schneller | Langsamer (mehr Berechnungen) |

In [15]:
### TODO: 2.2 Implement the train functionality

### YOUR CODE HERE

def train(model, dataloader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0

    optimizer.zero_grad()

    predictions = []
    ground_truth = []
    for inputs, labels, lengths in tqdm.tqdm(dataloader, desc="Train"):
        inputs = inputs.to(device)
        labels = labels.to(device)

        logits, hidden, weights = model(inputs, lengths)

        preds = torch.argmax(logits, axis=-1)

        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        running_loss += loss.item()
        predictions.extend(preds.cpu().numpy().tolist())
        ground_truth.extend(labels.cpu().numpy().tolist())

    running_loss = running_loss / len(dataloader)
    return predictions, ground_truth, running_loss

### END YOUR CODE

In [17]:
### TODO: 2.2 Implement the evaluation functionality

### YOUR CODE HERE

def eval(model, dataloader, criterion, device, return_attn_dict=False):
    model.eval()

    running_loss = 0.0

    predictions = []
    ground_truth = []
    sentences = []
    attn_weights = []
    with torch.no_grad():
        for inputs, labels, lengths in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            logits, hidden, weights = model(inputs, lengths)

            preds = torch.argmax(logits, axis=-1)

            loss = criterion(logits, labels)

            running_loss += loss.item()
            predictions.extend(preds.cpu().numpy().tolist())
            ground_truth.extend(labels.cpu().numpy().tolist())

            if return_attn_dict:
                sentences.append(inputs.cpu().numpy())
                attn_weights.append(weights.cpu().squeeze().numpy())

    running_loss = running_loss / len(dataloader)

    if return_attn_dict:
        attn_dict = {"tokens": sentences, "weights": attn_weights}
        return predictions, ground_truth, attn_dict
    else:
        return predictions, ground_truth, running_loss


def compute_metrics(preds, labels):
    return {
        'f1': sklearn_metrics.f1_score(y_true=labels, y_pred=preds),
        'prec': sklearn_metrics.precision_score(y_true=labels, y_pred=preds),
        'recall': sklearn_metrics.recall_score(y_true=labels, y_pred=preds),
        'acc': sklearn_metrics.accuracy_score(y_true=labels, y_pred=preds)
    }

### END YOUR CODE

In [20]:
### TODO: 2.3 Initialize and train the RNN Classification Model for X epochs + Evaluation

# Training parameters
SEED = 42
EPOCHS = 10
BATCH_SIZE = 16

LEARNING_RATE = 0.000001

DEVICE = "mps" # 'cpu', 'mps' or 'cuda'
LABEL_COL = "label"
PAD_IDX = word2idx["<pad>"]

# Model parameters
EMBEDDING_DIM = 256
HIDDEN_DIM = 256

### YOUR CODE HERE


folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(
    dataframe.tokenized, dataframe[LABEL_COL].values
)

VARIANTS = [("gru", False), ("gru", True)]


# Iterate variatants
results_all = []
models_all = {}
for rnn_variant, with_attention in VARIANTS:
    model_name = rnn_variant
    if with_attention:
        model_name += "-attention"

    # Iterate folds
    for fold, (train_idx, test_idx) in enumerate(folds, start=1):
        train_data = dataframe.iloc[train_idx]
        test_data = dataframe.iloc[test_idx]

        # Oversampling for minority class
        train_data_master = train_data[train_data["Grad"] == "Master"]
        train_data = pd.concat([train_data, pd.DataFrame(train_data_master.to_dict('records')  * 2)])

        # Prepare samples
        train_labels = train_data[LABEL_COL].values
        test_labels = test_data[LABEL_COL].values
        train_data = train_data.tokenized
        test_data = test_data.tokenized

        # Use higher batch_size for training
        train_dataset = ThesisClassificationDataset(train_data, train_labels, word2idx=word2idx)
        train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, collate_fn=SequencePadder(PAD_IDX))

        # Use batch_size=1 to avoid padding influence during testing
        test_dataset = ThesisClassificationDataset(test_data, test_labels, word2idx=word2idx)
        test_dataloader = DataLoader(test_dataset, batch_size=1, collate_fn=SequencePadder(PAD_IDX))

        model = GRU_Classifier(
            num_embeddings=len(vocab),
            embedding_dim=EMBEDDING_DIM,
            hidden_dim=HIDDEN_DIM,
            with_attention=with_attention,
            num_classes=2
        )
        model = model.to(DEVICE)

        criterion = nn.CrossEntropyLoss(reduction="mean")

        optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

        best_epoch = -1
        best_f1 = 0
        for epoch in range(1, EPOCHS + 1):
            print(f"Epoch {epoch} of {EPOCHS}")
            print("-" * 20)

            # Training step
            train_preds, train_labels, train_loss = train(
                model=model,
                dataloader=train_dataloader,
                criterion=criterion,
                optimizer=optimizer,
                device=DEVICE
            )

            # Evaluation step
            test_preds, test_labels, test_loss = eval(
                model=model,
                dataloader=test_dataloader,
                criterion=criterion,
                device=DEVICE
            )

            # Compute all metrics
            train_metrics = compute_metrics(preds=train_preds, labels=train_labels)
            test_metrics = compute_metrics(preds=test_preds, labels=test_labels)

            print(f"Train loss: {train_loss:.4f} | Train F1: {train_metrics['f1']:.4f}")
            print(f"Test loss: {test_loss:.4f} | Test F1: {test_metrics['f1']:.4f}")

            # Save best model by evaluation loss
            if best_f1 <= test_metrics["f1"]:
                best_epoch = epoch
                best_f1 = test_metrics["f1"]
                print(f"Saving best model ...")
                torch.save(model.state_dict(), f"data/best_{model_name}_clf.pt")

        # Load best and final model
        print(f"Best epoch: {best_epoch}")
        print(f"Best F1: {best_f1:.4f}")
        print(f"Loading best model ...")
        model.load_state_dict(torch.load(f"data/best_{model_name}_clf.pt"))

        # Compute test metrics
        test_preds, test_labels, test_loss = eval(
            model=model, dataloader=test_dataloader, criterion=criterion, device=DEVICE
        )
        test_metrics = compute_metrics(preds=test_preds, labels=test_labels)
        results_all.append({"variant": model_name, **test_metrics})
        print(test_metrics)

        models_all[model_name] = model

        # Remove for cross-validation results
        break

### END YOUR CODE

Epoch 1 of 10
--------------------


Train: 100%|██████████| 153/153 [00:06<00:00, 25.47it/s]


Train loss: 0.7010 | Train F1: 0.5118
Test loss: 0.7180 | Test F1: 0.2824
Saving best model ...
Epoch 2 of 10
--------------------


Train: 100%|██████████| 153/153 [00:05<00:00, 26.96it/s]


Train loss: 0.7001 | Train F1: 0.5128
Test loss: 0.7171 | Test F1: 0.2807
Epoch 3 of 10
--------------------


...

Train loss: 0.6898 | Train F1: 0.0000
Test loss: 0.6631 | Test F1: 0.1101
Epoch 9 of 10
--------------------


Train: 100%|██████████| 153/153 [00:06<00:00, 22.29it/s]


Train loss: 0.6897 | Train F1: 0.0000
Test loss: 0.6625 | Test F1: 0.1101
Epoch 10 of 10
--------------------


Train: 100%|██████████| 153/153 [00:06<00:00, 23.06it/s]


Train loss: 0.6895 | Train F1: 0.0000
Test loss: 0.6619 | Test F1: 0.1101
Best epoch: 3
Best F1: 0.1250
Loading best model ...
{'f1': 0.125, 'prec': 0.3333333333333333, 'recall': 0.07692307692307693, 'acc': 0.7694117647058824}


In [21]:
results_df = pd.DataFrame(results_all)
results_df = results_df.groupby("variant").mean()
results_df

,f1,prec,recall,acc
variant,,,,
gru,0.283186,0.194332,0.521739,0.429577
gru-attention,0.125000,0.333333,0.076923,0.769412


In [12]:
### TODO: 2.4 Visualize the attention weights

### YOUR CODE HERE

import random
import matplotlib
from IPython.display import display, HTML


def colorize_sentence(words, color_array, cmap = matplotlib.cm.get_cmap('RdBu')):
    assert(len(words) == len(color_array))
    # color_array is an array of numbers between 0 and 1 of length equal to words
    template = '<span class="barcode"; style="color: black; background-color: {}">{}</span>'
    colored_string = ''
    for word, color in zip(words, color_array):
        color = matplotlib.colors.rgb2hex(cmap(color.item())[:3])
        colored_string += template.format(color, '&nbsp' + word + '&nbsp')
    return colored_string


# Get predictions and attention weights
test_dataset = ThesisClassificationDataset(test_data, test_labels, word2idx=word2idx)
test_dataloader = DataLoader(test_dataset, batch_size=1, collate_fn=SequencePadder(PAD_IDX))
preds, labels, attn_dict = eval(
    model=models_all["gru-attention"], dataloader=test_dataloader,  device=DEVICE,
    criterion=nn.CrossEntropyLoss(reduction="mean"), return_attn_dict=True
)


# Make word color maps
attn_weights = []
colorized_sentences = []
for tokens, weights in zip(attn_dict["tokens"], attn_dict["weights"]):
    words = [idx2word[token[0]] for token in tokens]
    weights = np.array(weights) / np.max(weights)
    attn_weights.append(weights)
    colorized_sentences.append(colorize_sentence(words, weights))

# Display 10 random sentences
for i in [random.randint(0, len(colorized_sentences)) for _ in range(10)]:
    # print(attn_weights[i])
    display(HTML(colorized_sentences[i]))

### END YOUR CODE

/var/folders/fz/9kmbj46x27q5mz39hqvmh3x40000gn/T/ipykernel_9815/690638433.py:8: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  def colorize_sentence(words, color_array, cmap = matplotlib.cm.get_cmap('RdBu')):
